In [ ]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
from scipy.stats import genpareto
from biascorrect import load_pairs,fit_qm,apply_qm
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR = CONFIGS['filepaths']['splits']
PREDSDIR  = CONFIGS['filepaths']['predictions']
MODELS    = CONFIGS['experiments']
LATRANGE  = CONFIGS['domain']['latrange']
LONRANGE  = CONFIGS['domain']['lonrange']
SPLIT      = 'test'
NQUANTILES = 200
TAILPCT    = 95
QUANTILES  = {50:'#1B2C61',100:'#5BA7DA',200:'#F2C85E',500:'#D42028'}

MODELDICT = {}
ORDER     = []
for name,rc in MODELS['sr']['optimizedeqs'].items():
    predpath = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if not os.path.exists(predpath):
        continue
    MODELDICT[name] = {'label':rc['description'],'color':rc['color'],'marker':'o'}
    ORDER.append(name)
FITSPLITS  = ['train','valid']
FIGSDIR    = 'figs'
os.makedirs(FIGSDIR,exist_ok=True)

In [ ]:
era5,imerg = load_pairs(SPLITSDIR,FITSPLITS)
mapping    = fit_qm(era5,imerg,nquantiles=NQUANTILES)
print(f'Fit quantile mapping on {len(era5):,} paired samples (nquantiles = {NQUANTILES})')

In [ ]:
era5pos = era5[era5>0]
tail    = np.quantile(era5pos,TAILPCT/100)
xeval   = np.linspace(0,era5pos.max(),1000)

fig,ax  = pplt.subplots(figwidth=2.5)
ax.format(xlabel='ERA5 Total Precipitation (mm)',xlim=(1e-2,140),xticks=[0.01,1,100],xscale='log',xformatter='log',
          ylabel='IMERG V06 Precipitation Rate (mm/hr)',ylim=(1e-2,140),yticks=[0.01,1,100],yscale='log',yformatter='log',
          grid=False)
for nquantiles,color in QUANTILES.items():
    m = fit_qm(era5,imerg,nquantiles=nquantiles)
    ax.plot(xeval,m(xeval),label=fr'$\mathit{{N}} = {nquantiles}$',color=color,alpha=0.7,linewidth=1.5)
ax.axvspan(tail,era5pos.max(),color='gray',alpha=0.2,label=fr'$> P_{{{TAILPCT}}}$',zorder=0)
ax.axvline(tail,color='gray',linestyle=':',linewidth=1)
ax.legend(loc='ul',ncols=1)
pplt.show()
fig.save(os.path.join(FIGSDIR,'fig_S2.jpg'))

In [ ]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    trueimerg = ds.pr.load()

results = {}
for name in MODELDICT:
    filepath = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if not os.path.exists(filepath):
        continue
    with xr.open_dataset(filepath) as ds:
        predtp = ds.tp.load()
    if 'seed' in predtp.dims:
        predtp = predtp.mean('seed')
    if 'complexity' in predtp.dims:
        predtp = predtp.isel(complexity=0)
    predpr      = apply_qm(mapping,predtp.squeeze())
    ytrue,ypred = xr.align(trueimerg,predpr,join='inner')
    results[name] = (ytrue.squeeze(),ypred.squeeze())

print(f'Loaded {len(results)}/{len(MODELDICT)} models (bias-corrected to IMERG V06)')

In [ ]:
def exceedance_curve(arr):
    flat = arr.ravel()
    flat = flat[np.isfinite(flat)&(flat>0)]
    flat = np.sort(flat)
    prob = np.arange(len(flat),0,-1)/len(flat)
    return flat,prob

obs = next(iter(results.values()))[0]
obsvalues,obsprob = exceedance_curve(obs.values)

In [ ]:
fig,ax = pplt.subplots(figwidth=2.5)
ax.plot(obsvalues,obsprob,color='k',linestyle='--',linewidth=2,label='Observed',zorder=0)
for name in ORDER:
    if name not in results:
        continue
    info = MODELDICT[name]
    predvalues,predprob = exceedance_curve(results[name][1].values)
    ax.plot(predvalues,predprob,color=info['color'],linewidth=1.5,label=info['label'])
ax.format(grid=False,xlabel='Precipitation Rate (mm/hr)',xscale='log',xformatter='log',
          ylabel='Exceedance Probability',yscale='log',yformatter='log')
ax.legend(loc='ll',ncols=1)
pplt.show()
fig.save(os.path.join(FIGSDIR,'fig_6.jpg'))

In [ ]:
ordered = [name for name in ORDER if name in results]
biases  = []
for name in ordered:
    ytrue,ypred = results[name]
    obsp95  = ytrue.quantile(0.95,dim='time')
    predp95 = ypred.quantile(0.95,dim='time')
    biases.append(predp95-obsp95)

ncols = min(len(ordered),4)
nrows = (len(ordered)+ncols-1)//ncols
fig,axs = pplt.subplots(nrows=nrows,ncols=ncols,proj='cyl',figwidth=10,share=True)
axs.format(coast=True,latlim=LATRANGE,lonlim=LONRANGE,latlines=[10,15,20],lonlines=[65,75,85],grid=False)
axs[-1,:].format(lonlabels='b')
axs[:,0].format(latlabels='l')
m = None
for ax,name,bias in zip(axs,ordered,biases):
    m = ax.pcolormesh(bias.lon,bias.lat,bias,cmap='ColdHot',vmin=-2,vmax=2,levels=12,extend='both')
    ax.format(title=MODELDICT[name]['label'])
for ax in axs[len(ordered):]:
    ax.set_visible(False)
fig.colorbar(m,loc='b',label='Bias-Corrected $P_{95}$ Bias (Model $-$ Observed) (mm/hr)',ticks=1)
axs.format(abc=True,titleloc='l')
pplt.show()
fig.save(os.path.join(FIGSDIR,'fig_S3.jpg'))